# FraudShieldAI: Intelligent Real-Time Banking Fraud Detection
### IBM Z Datathon 2026 — IBM LinuxONE (s390x) Environment

**Problem Statement:** In digital banking, fraud detection must occur before funds settle, identifying anomalies with low latency, assigning calibrated risk scores, and preventing high-risk demo transactions from automatic approval.

> **ETHICAL & SIMULATION NOTICE:** This system runs on 100% synthetically generated banking transactions. It does not collect or process real credit cards, PII, actual bank credentials, or live payment network data.

## Step 1: Environment & IBM LinuxONE Hardware Verification
Confirming Python 3.12 and the native IBM Z `s390x` architecture.

In [ ]:
import platform, sys, os
import numpy as np
import pandas as pd
import sklearn
import joblib

print('=====================================================')
print('  FraudShieldAI — IBM LinuxONE Environment Verification')
print('=====================================================')
print(f'System Platform : {platform.system()} {platform.release()}')
print(f'Architecture    : {platform.machine()} (Byte Order: {sys.byteorder.upper()})')
print(f'Python Version  : {sys.version.split()[0]}')
print(f'Scikit-Learn    : {sklearn.__version__}')
print(f'Pandas Version  : {pd.__version__}')
print(f'NumPy Version   : {np.__version__}')

## Step 2: Load Synthetic Banking Dataset
Loading the 5,000 synthetic transaction records generated for this prototype.

In [ ]:
csv_path = 'data/synthetic_transactions.csv'
if not os.path.exists(csv_path):
    csv_path = '../data/synthetic_transactions.csv'
df = pd.read_csv(csv_path)

print(f'Total transactions loaded: {len(df):,}')
print(f'Legitimate transactions : {(df["is_fraud"] == 0).sum():,} ({(df["is_fraud"] == 0).mean()*100:.1f}%)')
print(f'Fraudulent transactions : {(df["is_fraud"] == 1).sum():,} ({(df["is_fraud"] == 1).mean()*100:.1f}%)')
df.head(5)

## Step 3: Train-Test Split and Preprocessing Pipeline
Separating 80% for training and 20% for out-of-sample evaluation using stratified sampling.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

numeric_features = [
    'amount', 'hour_of_day', 'day_of_week', 'tx_velocity_1h',
    'avg_amount_ratio', 'is_international', 'device_trust_score',
    'ip_risk_score', 'distance_from_home', 'failed_logins_24h',
    'is_first_merchant_use'
]
categorical_features = ['merchant_category']

X = df[numeric_features + categorical_features]
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

rf_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=4,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', rf_classifier)
])

print(f'Training set records: {len(X_train):,}')
print(f'Testing set records : {len(X_test):,}')

## Step 4: Model Training on IBM LinuxONE (s390x)
Fitting the Random Forest model with balanced weights.

In [ ]:
pipeline.fit(X_train, y_train)
print('Model training completed successfully on IBM LinuxONE!')

## Step 5: Evaluation Metrics (Precision, Recall, F1, ROC-AUC)
Validating classification discrimination on the holdout test set.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred)

print(f'ROC-AUC Score: {roc_auc:.4f}')
print('\nConfusion Matrix:')
print(f'  True Negatives (Legitimate Approved) : {cm[0, 0]}')
print(f'  False Positives (Legitimate Flagged) : {cm[0, 1]}')
print(f'  False Negatives (Fraud Missed)       : {cm[1, 0]}')
print(f'  True Positives (Fraud Caught)        : {cm[1, 1]}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Legitimate (0)', 'Fraud (1)']))

## Step 6: Global Feature Importance Ranking
Analyzing top drivers for explainability in banking compliance.

In [ ]:
rf = pipeline.named_steps['classifier']
cat_encoder = pipeline.named_steps['preprocessor'].named_transformers_['cat']
cat_feature_names = list(cat_encoder.get_feature_names_out(categorical_features))
all_feature_names = numeric_features + cat_feature_names

importances = rf.feature_importances_
feat_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

print('Top 10 Fraud Prediction Drivers:')
for idx, row in feat_df.head(10).iterrows():
    print(f'  {idx+1:2d}. {row["Feature"]:22s} : {row["Importance"]*100:5.2f}%')

## Step 7: Explainable Risk Decision Simulation
Testing safe and fraudulent transactions through our composite risk decision matrix.

In [ ]:
import json

def evaluate_transaction(tx_dict):
    tx_df = pd.DataFrame([tx_dict])
    ml_prob = pipeline.predict_proba(tx_df)[0, 1]
    
    # Heuristic penalty calculation
    penalties = 0
    reasons = []
    if tx_dict['ip_risk_score'] > 0.7:
        penalties += 20; reasons.append('High-risk IP address / Proxy detected')
    if tx_dict['device_trust_score'] < 0.3:
        penalties += 20; reasons.append('Untrusted or newly seen device')
    if tx_dict['failed_logins_24h'] >= 2:
        penalties += 20; reasons.append(f"{tx_dict['failed_logins_24h']} failed logins in 24 hours")
    if tx_dict['avg_amount_ratio'] > 4.0:
        penalties += 25; reasons.append(f"Spending spike: {tx_dict['avg_amount_ratio']}x historical average")
    if tx_dict['distance_from_home'] > 300.0:
        penalties += 15; reasons.append(f"Geographic anomaly: {tx_dict['distance_from_home']} miles from home")
        
    # Composite score formula: 70% ML probability + 30% heuristic penalties (capped at 100)
    composite_score = round(min(100.0, (ml_prob * 70.0) + (penalties * 0.30)))
    
    if composite_score < 35:
        verdict = 'APPROVED (Low Risk)'
    elif composite_score < 70:
        verdict = 'FLAGGED_OTP (Medium Risk - Step-Up Auth Required)'
    else:
        verdict = 'HELD_FOR_REVIEW (High Risk - Admin Approval Required)'
        
    return {
        'Composite_Score': composite_score,
        'ML_Probability': f'{ml_prob*100:.1f}%',
        'Verdict': verdict,
        'Risk_Drivers': reasons if reasons else ['Normal transaction parameters']
    }

safe_sample = {
    'amount': 42.50, 'hour_of_day': 13, 'day_of_week': 1, 'merchant_category': 'grocery',
    'tx_velocity_1h': 1, 'avg_amount_ratio': 1.0, 'is_international': 0,
    'device_trust_score': 0.95, 'ip_risk_score': 0.05, 'distance_from_home': 4.0,
    'failed_logins_24h': 0, 'is_first_merchant_use': 0
}

fraud_sample = {
    'amount': 3800.00, 'hour_of_day': 3, 'day_of_week': 6, 'merchant_category': 'crypto',
    'tx_velocity_1h': 6, 'avg_amount_ratio': 7.5, 'is_international': 1,
    'device_trust_score': 0.05, 'ip_risk_score': 0.95, 'distance_from_home': 1200.0,
    'failed_logins_24h': 3, 'is_first_merchant_use': 1
}

print('=== Safe Demo Transaction Evaluation ===')
print(json.dumps(evaluate_transaction(safe_sample), indent=2))

print('\n=== High-Risk Demo Transaction Evaluation ===')
print(json.dumps(evaluate_transaction(fraud_sample), indent=2))